# Lecture 9: Practical Computation & Gram-Schmidt Stability
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.  
The original lecture focuses on MATLAB; here everything is translated to the Python/NumPy ecosystem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

## CGS and MGS Implementations

In [ ]:
def classical_gs(A):
    """Classical Gram-Schmidt (projects against original a_j)."""
    A = A.copy()
    m, n = A.shape
    Q = np.zeros((m, n), dtype=A.dtype)
    R = np.zeros((n, n), dtype=A.dtype)
    for j in range(n):
        v = A[:, j].copy()
        for i in range(j):
            R[i, j] = Q[:, i] @ A[:, j]
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        Q[:, j] = v / R[j, j]
    return Q, R

def modified_gs(A):
    """Modified Gram-Schmidt (projects against updated v)."""
    A = A.copy()
    m, n = A.shape
    Q = np.zeros((m, n), dtype=A.dtype)
    R = np.zeros((n, n), dtype=A.dtype)
    for j in range(n):
        R[j, j] = np.linalg.norm(A[:, j])
        Q[:, j] = A[:, j] / R[j, j]
        for k in range(j + 1, n):
            R[j, k] = Q[:, j] @ A[:, k]
            A[:, k] -= R[j, k] * Q[:, j]
    return Q, R

## Experiment 1: Diagonal of $R$ Tracks Singular Values

Build an $80 \times 80$ matrix with exponentially decaying singular values $\sigma_j = 2^{-j}$. The diagonal of $R$ should track these — but CGS and MGS diverge at different levels.

In [ ]:
n = 80
U, _, Vt = np.linalg.svd(np.random.randn(n, n))
s = 2.0 ** np.arange(-1, -(n + 1), -1)
A = U @ np.diag(s) @ Vt

Qc, Rc = classical_gs(A)
Qm, Rm = modified_gs(A)

plt.figure(figsize=(9, 5))
plt.semilogy(range(1, n+1), s, '.', markersize=8, label='Singular values')
plt.semilogy(range(1, n+1), np.abs(np.diag(Rc)), 'o', markersize=4, label='CGS diag($R$)')
plt.semilogy(range(1, n+1), np.abs(np.diag(Rm)), 's', markersize=4, label='MGS diag($R$)')
plt.axhline(np.finfo(np.float64).eps, color='gray', linestyle='--', alpha=0.5, label=r'$\epsilon_{\mathrm{machine}}$ (float64)')
plt.axhline(np.sqrt(np.finfo(np.float64).eps), color='gray', linestyle=':', alpha=0.5, label=r'$\sqrt{\epsilon_{\mathrm{machine}}}$')
plt.xlabel('$j$')
plt.ylabel(r'$\sigma_j$ or $|r_{jj}|$')
plt.title('Gram-Schmidt in double precision (float64)')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

### Single Precision (float32)

Repeat with `float32` ($\epsilon \approx 10^{-7}$). The same pattern holds at the new precision level.

In [ ]:
A32 = A.astype(np.float32)

Qc32, Rc32 = classical_gs(A32)
Qm32, Rm32 = modified_gs(A32)

plt.figure(figsize=(9, 5))
plt.semilogy(range(1, n+1), s, '.', markersize=8, label='Singular values')
plt.semilogy(range(1, n+1), np.abs(np.diag(Rc32)), 'o', markersize=4, label='CGS diag($R$)')
plt.semilogy(range(1, n+1), np.abs(np.diag(Rm32)), 's', markersize=4, label='MGS diag($R$)')
plt.axhline(np.finfo(np.float32).eps, color='gray', linestyle='--', alpha=0.5, label=r'$\epsilon_{\mathrm{machine}}$ (float32)')
plt.axhline(np.sqrt(np.finfo(np.float32).eps), color='gray', linestyle=':', alpha=0.5, label=r'$\sqrt{\epsilon_{\mathrm{machine}}}$')
plt.xlabel('$j$')
plt.ylabel(r'$\sigma_j$ or $|r_{jj}|$')
plt.title('Gram-Schmidt in single precision (float32)')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"float32 eps = {np.finfo(np.float32).eps:.2e}")
print(f"sqrt(eps)   = {np.sqrt(np.finfo(np.float32).eps):.2e}")

## Experiment 2: Loss of Orthogonality (Nearly Parallel Columns)

When columns are nearly parallel, even MGS loses orthogonality due to catastrophic cancellation.

In [ ]:
A = np.array([[np.pi,      np.sqrt(2)],
              [355.0/113,  np.sqrt(2)]])

print(f"pi           = {np.pi:.16f}")
print(f"355/113      = {355/113:.16f}")
print(f"Difference   = {np.pi - 355/113:.6e}")
print(f"cond(A)      = {np.linalg.cond(A):.2e}\n")

# MGS orthogonalization
q1 = A[:, 0] / np.linalg.norm(A[:, 0])
v = A[:, 1] - (q1 @ A[:, 1]) * q1
print(f"Remainder after projection: {v}")
print(f"Magnitude: {np.linalg.norm(v):.6e}  (tiny!)")

q2 = v / np.linalg.norm(v)
Q = np.column_stack([q1, q2])
orth_err = np.linalg.norm(Q.T @ Q - np.eye(2))
print(f"\n||Q^TQ - I|| = {orth_err:.6e}  (far from machine eps!)")

## Floating-Point Realities: Catastrophic Cancellation

In [ ]:
print(f"(1 + 1e-16) - 1 = {(1 + 1e-16) - 1}   (should be 1e-16)")
print(f"(1 + 1e-15) - 1 = {(1 + 1e-15) - 1}   (should be 1e-15)")
print(f"(1 + 1e-10) - 1 = {(1 + 1e-10) - 1}   (should be 1e-10)")
print(f"\nMachine epsilon (float64): {np.finfo(np.float64).eps:.2e}")
print(f"Machine epsilon (float32): {np.finfo(np.float32).eps:.2e}")

## Householder QR: Machine-Precision Orthogonality

In [ ]:
np.random.seed(0)
A = np.random.randn(100, 50)

Q_hh, R_hh = np.linalg.qr(A)
Q_mgs, R_mgs = modified_gs(A)
Q_cgs, R_cgs = classical_gs(A)

print(f"cond(A) = {np.linalg.cond(A):.2e}\n")
print(f"Householder: ||Q^TQ - I|| = {np.linalg.norm(Q_hh.T @ Q_hh - np.eye(50)):.2e}")
print(f"MGS:         ||Q^TQ - I|| = {np.linalg.norm(Q_mgs.T @ Q_mgs - np.eye(50)):.2e}")
print(f"CGS:         ||Q^TQ - I|| = {np.linalg.norm(Q_cgs.T @ Q_cgs - np.eye(50)):.2e}")

## The Hilbert Matrix: Ill-Conditioning in Action

In [ ]:
from scipy.linalg import hilbert

sizes = range(2, 21)
conds = [np.linalg.cond(hilbert(n)) for n in sizes]

plt.figure(figsize=(8, 4.5))
plt.semilogy(list(sizes), conds, 'ro-', markersize=5)
plt.axhline(1 / np.finfo(float).eps, color='gray', linestyle='--',
            alpha=0.5, label=r'$1/\epsilon_{\mathrm{machine}}$')
plt.xlabel('Matrix size $n$')
plt.ylabel(r'$\kappa(H_n)$')
plt.title('Condition number of the Hilbert matrix')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

# Demonstrate loss of accuracy
for n in [5, 10, 15, 20]:
    H = hilbert(n)
    x_true = np.ones(n)
    b = H @ x_true
    x_comp = np.linalg.solve(H, b)
    rel_err = np.linalg.norm(x_comp - x_true) / np.linalg.norm(x_true)
    print(f"n={n:2d}: cond = {np.linalg.cond(H):.2e}, relative error = {rel_err:.2e}")

## Least Squares: Four Ways (Only Three Are Good)

In [ ]:
np.random.seed(1)
m, n = 100, 20
A = np.random.randn(m, n)
b = np.random.randn(m)

# 1. lstsq (auto)
x1 = np.linalg.lstsq(A, b, rcond=None)[0]

# 2. Explicit QR
Q, R = np.linalg.qr(A)
x2 = np.linalg.solve(R, Q.T @ b)

# 3. Pseudoinverse (SVD)
x3 = np.linalg.pinv(A) @ b

# 4. Normal equations (DON'T do this)
x4 = np.linalg.solve(A.T @ A, A.T @ b)

print(f"cond(A)    = {np.linalg.cond(A):.2e}")
print(f"cond(A^TA) = {np.linalg.cond(A.T @ A):.2e}  (squared!)\n")

print(f"||x1 - x2|| = {np.linalg.norm(x1 - x2):.2e}  (lstsq vs QR)")
print(f"||x1 - x3|| = {np.linalg.norm(x1 - x3):.2e}  (lstsq vs pinv)")
print(f"||x1 - x4|| = {np.linalg.norm(x1 - x4):.2e}  (lstsq vs normal eqs)")

## Normal Equations Fail on Ill-Conditioned Problems

In [ ]:
np.random.seed(2)
m, n = 50, 10

# Build a matrix with large condition number
U, _ = np.linalg.qr(np.random.randn(m, n))
V, _ = np.linalg.qr(np.random.randn(n, n))
s = np.logspace(0, -8, n)  # kappa ~ 1e8
A = U @ np.diag(s) @ V.T
x_true = np.random.randn(n)
b = A @ x_true

# QR approach
Q, R = np.linalg.qr(A)
x_qr = np.linalg.solve(R, Q.T @ b)

# Normal equations
x_ne = np.linalg.solve(A.T @ A, A.T @ b)

print(f"cond(A)    = {np.linalg.cond(A):.2e}")
print(f"cond(A^TA) = {np.linalg.cond(A.T @ A):.2e}\n")
print(f"QR:     ||x - x_true|| / ||x_true|| = {np.linalg.norm(x_qr - x_true) / np.linalg.norm(x_true):.2e}")
print(f"Normal: ||x - x_true|| / ||x_true|| = {np.linalg.norm(x_ne - x_true) / np.linalg.norm(x_true):.2e}")

## Timing: QR Complexity $\sim \frac{2}{3} n^3$

In [ ]:
sizes = [100, 200, 500, 1000, 2000]
times_qr = []
times_svd = []

for n in sizes:
    A = np.random.randn(n, n)

    t0 = time.perf_counter()
    np.linalg.qr(A)
    times_qr.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    np.linalg.svd(A)
    times_svd.append(time.perf_counter() - t0)

plt.figure(figsize=(8, 5))
plt.loglog(sizes, times_qr, 'bo-', label='QR')
plt.loglog(sizes, times_svd, 'rs-', label='SVD')

# n^3 reference
ref = [(n / sizes[-1])**3 * times_qr[-1] for n in sizes]
plt.loglog(sizes, ref, 'k--', alpha=0.5, label=r'$O(n^3)$ reference')

plt.xlabel('$n$')
plt.ylabel('Time (s)')
plt.title('Factorization timing')
plt.legend()
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

for n, tq, ts in zip(sizes, times_qr, times_svd):
    print(f"n={n:5d}: QR = {tq:.4f}s, SVD = {ts:.4f}s")

## Dense vs. Sparse

In [ ]:
from scipy import sparse
from scipy.sparse.linalg import spsolve

n = 5000
density = 0.001
A_sp = sparse.random(n, n, density=density, format='csc') + sparse.eye(n) * 10
b = np.random.randn(n)

# Sparse solve
t0 = time.perf_counter()
x_sp = spsolve(A_sp, b)
t_sparse = time.perf_counter() - t0

# Dense solve
A_dense = A_sp.toarray()
t0 = time.perf_counter()
x_dense = np.linalg.solve(A_dense, b)
t_dense = time.perf_counter() - t0

print(f"Matrix size: {n}x{n}")
print(f"Nonzeros:    {A_sp.nnz} ({100*A_sp.nnz/n**2:.2f}%)")
print(f"Dense solve: {t_dense:.4f}s")
print(f"Sparse solve: {t_sparse:.4f}s")
print(f"Speedup:     {t_dense/t_sparse:.1f}x")
print(f"||x_sparse - x_dense|| = {np.linalg.norm(x_sp - x_dense):.2e}")

In [ ]:
# Visualize sparsity pattern (smaller matrix for clarity)
A_small = sparse.random(200, 200, density=0.01, format='csc')

plt.figure(figsize=(5, 5))
plt.spy(A_small, markersize=1)
plt.title(f'Sparsity pattern ({A_small.nnz} nonzeros in 200x200)')
plt.tight_layout()
plt.show()

## Norms and Condition Numbers in NumPy

In [ ]:
np.random.seed(5)
x = np.random.randn(10)
A = np.random.randn(10, 10)

print("Vector norms:")
print(f"  ||x||_1   = {np.linalg.norm(x, 1):.4f}")
print(f"  ||x||_2   = {np.linalg.norm(x):.4f}")
print(f"  ||x||_inf = {np.linalg.norm(x, np.inf):.4f}")

print("\nMatrix norms:")
print(f"  ||A||_2   = {np.linalg.norm(A, 2):.4f}")
print(f"  ||A||_fro = {np.linalg.norm(A, 'fro'):.4f}")

print(f"\nCondition numbers:")
print(f"  cond(randn(10x10))  = {np.linalg.cond(A):.2f}")

from scipy.linalg import hilbert
for n in [5, 10, 15, 20]:
    print(f"  cond(hilbert({n:2d})) = {np.linalg.cond(hilbert(n)):.2e}")